<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/vima/Kettos%20Kompetencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Szükséges Importok**

In [12]:
# ═══════════════════════════════════════════════════════════════════
# NE MÓDOSÍTSD: ezzel a seeddel kapod ugyanazt az eredményt minden
# újrafuttatáskor. A szerver determinisztikus.
# ═══════════════════════════════════════════════════════════════════
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

# --- Reprodukálhatóság: minden véletlent egyetlen seedhez kötünk ---
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# A feladat megoldásához GPU ajánlott
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

device: cuda


## **Fájlok letöltése**

In [13]:
import os
import gdown

if not os.path.isdir("data"):
    gdown.download_folder(id="1FU9egPtbJYDUKpIBEh8BIV0zwVQ5Zqf5", output="data")

## **Modell architektúra**

In [14]:
class ConvVAE(nn.Module):
    def __init__(self):
        super(ConvVAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.enc_mu = nn.Linear(64*8*8, 16)
        self.enc_logvar = nn.Linear(64*8*8, 16)

        self.decoder = nn.Sequential(
            nn.Linear(16, 64*8*8),
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        enc = self.encoder(x)

        mu = self.enc_mu(enc)
        logvar = self.enc_logvar(enc)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        z = mu + eps * std

        return self.decoder(z), mu, logvar, enc

## **Fájlok betöltése**

In [15]:
model = ConvVAE().to(device)
model.load_state_dict(torch.load("data/ae.pt", map_location=device))

transform = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])
cifar_labeled_ds = datasets.CIFAR10('./data', train=True, download=True, transform=transform)

cifar_tensors = torch.stack([x for x, _ in cifar_labeled_ds])
cifar_ds = TensorDataset(cifar_tensors)
train_loader = DataLoader(cifar_ds, batch_size=128, shuffle=True)

## **Itt kezdődik a te kódod**

In [19]:
import torch.nn.functional as F

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

cifar_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

mnist_loader = DataLoader(mnist_test, batch_size=128, shuffle=False)
cifar_loader = DataLoader(cifar_test, batch_size=128, shuffle=False)

@torch.no_grad()
def compute_mse(model, loader):
    model.eval()

    total_loss = 0.0
    total_pixels = 0

    for x, _ in loader:
        x = x.to(device)

        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        recon, _, _, _ = model(x)
        # recon = model.reconstruct(x)

        mse = F.mse_loss(recon, x, reduction="sum")

        total_loss += mse.item()
        total_pixels += x.numel()

    return total_loss / total_pixels


def evaluate_model(model, BMNIST=0.009, BCIFAR=0.0100, L_nop_CIFAR=0.0350):
    L_MNIST = compute_mse(model, mnist_loader)
    L_CIFAR = compute_mse(model, cifar_loader)

    f_MNIST = max(
        0.0,
        1.0 - 0.9 * ((L_MNIST / BMNIST) - 1.0) / 2.0
    )

    f_CIFAR = ((L_nop_CIFAR - L_CIFAR)/ (L_nop_CIFAR - BCIFAR))

    f_CIFAR = max(0.0, min(1.0, f_CIFAR))
    score = f_MNIST * f_CIFAR

    stats = {
        "L_MNIST": L_MNIST,
        "L_CIFAR": L_CIFAR,
        "f_MNIST": f_MNIST,
        "f_CIFAR": f_CIFAR,
        "score": score
    }

    return score, stats

In [ ]:
import copy

model = ConvVAE().to(device)
model.load_state_dict(torch.load("data/ae.pt", map_location=device))
reference_model = copy.deepcopy(model).to(device)
reference_model.eval()
original_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
recon_loss = nn.MSELoss()

def stability_loss(model, reference_model, x):
    with torch.no_grad():
        ref_out, _, _, ref_enc = reference_model(x)

    out, _, _, enc = model(x)

    return F.mse_loss(out, ref_out) + F.mse_loss(enc, ref_enc)
# def stability_regularization(model, original_state):
#     reg = 0.0
#     for name, param in model.named_parameters():
#         reg += ((param - original_state[name]) ** 2).mean()

#     return reg

def train(model, epochs, optimizer, lambda_reg=0.1):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_loss_reg = 0
        total_loss_recon = 0
        for x in train_loader:
            x = x[0].to(device)
            recon, mu, logvar, _ = model(x)
            loss_recon = recon_loss(recon, x)
            # loss_reg = stability_regularization(model, original_state) * lambda_reg
            loss_reg = stability_loss(model, reference_model, x) * lambda_reg
            loss = loss_recon + loss_reg
            optimizer.zero_grad()
            total_loss += loss.item()
            total_loss_reg += loss_reg.item()
            total_loss_recon += loss_recon.item()
            loss.backward()
            optimizer.step()
        score, stats = evaluate_model(model)
        print(f"Epoch: {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Recon: {total_loss_recon/len(train_loader)} | Reg: {total_loss_reg/len(train_loader)} | Score: {score:.4f} | " +
              f"L_MNIST: {stats['L_MNIST']:.4f} | L_CIFAR: {stats['L_CIFAR']:.4f} | F_CIFAR: {stats['f_CIFAR']:.4f} | F_MNIST: {stats['f_MNIST']:.4f}")    


In [28]:

model = ConvVAE().to(device)
model.load_state_dict(torch.load("data/ae.pt", map_location=device))

evaluate_model(model)

(0.0,
 {'L_MNIST': 0.009023014370600383,
  'L_CIFAR': 0.22885488335291546,
  'f_MNIST': 0.9988492814699808,
  'f_CIFAR': 0.0,
  'score': 0.0})

In [ ]:
from torch import optim

model = ConvVAE().to(device)
model.load_state_dict(torch.load("data/ae.pt", map_location=device))
for param in model.encoder.parameters():
    param.requires_grad = False
    

optimizer = optim.AdamW(params=model.parameters())
train(model, 50, optimizer, 0.2)

Epoch: 1 | Loss: 0.1433 | Recon: 0.08065156058391647 | Reg: 0.02961031473039285 | Score: 0.0000 | L_MNIST: 0.1609 | L_CIFAR: 0.0687 | F_CIFAR: 0.0000 | F_MNIST: 0.0000
Epoch: 2 | Loss: 0.0980 | Recon: 0.06702043163730666 | Reg: 0.029513397432692216 | Score: 0.0000 | L_MNIST: 0.1624 | L_CIFAR: 0.0681 | F_CIFAR: 0.0000 | F_MNIST: 0.0000
Epoch: 3 | Loss: 0.0967 | Recon: 0.06667464813384254 | Reg: 0.02943312393410889 | Score: 0.0000 | L_MNIST: 0.1698 | L_CIFAR: 0.0663 | F_CIFAR: 0.0000 | F_MNIST: 0.0000
Epoch: 4 | Loss: 0.0963 | Recon: 0.06648750289741075 | Reg: 0.029391205833886592 | Score: 0.0000 | L_MNIST: 0.1681 | L_CIFAR: 0.0667 | F_CIFAR: 0.0000 | F_MNIST: 0.0000
Epoch: 5 | Loss: 0.0960 | Recon: 0.0663972929634554 | Reg: 0.029386780079444657 | Score: 0.0000 | L_MNIST: 0.1679 | L_CIFAR: 0.0666 | F_CIFAR: 0.0000 | F_MNIST: 0.0000


In [ ]:
evaluate_model(model)

(0.0,
 {'L_MNIST': 0.03490284014542898,
  'L_CIFAR': 0.016723526565233866,
  'f_MNIST': 0.0,
  'f_CIFAR': 0.7310589373906454,
  'score': 0.0})